In [1]:
import sys
import os
import requests
import json
sys.path.append(os.path.abspath(".."))
from sqlalchemy import create_engine, select, text, Column, BigInteger, Text, TIMESTAMP, ForeignKey, func
from sqlalchemy.orm import declarative_base, sessionmaker, relationship
from pgvector.sqlalchemy import Vector
from sqlalchemy.orm import sessionmaker
from sqlalchemy.exc import IntegrityError
from db.models import Topic, Subtopic, DocumentChunk
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
import time

c:\Users\Phyo Sandar Win\Desktop\FYP\Source Code\AY2025-Atlas-NALA-FYP\chatbot-backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
start_time = time.time()
embedding_model = SentenceTransformer('BAAI/bge-m3')

print("BGE Embedding model loaded in --- %s seconds ---" % (time.time() - start_time))

BGE Embedding model loaded in --- 10.065861225128174 seconds ---


In [7]:
from FlagEmbedding import FlagReranker
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

In [8]:
load_dotenv()
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASS = os.getenv("POSTGRES_PASS")
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")

# Initialise database URL
database_url = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASS}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(database_url)
Session = sessionmaker(bind=engine)
session = Session()

NALA_API_KEY = os.getenv("NALA_API_KEY")
BASE_URL = os.getenv("BASE_URL")

In [9]:
import time

def call_gemini_rag(context: str, user_question: str, retries: int = 3, delay: int = 5):
    """
    Calls the Gemini LLM API with retry logic in case of failure.

    Args:
        context (str): The context to provide to the LLM.
        user_question (str): The user's question.
        retries (int): Number of retry attempts in case of failure.
        delay (int): Time delay (in seconds) between retries.

    Returns:
        str: The response from the Gemini LLM API or an error message.
    """
    url = f"{BASE_URL}/api/llm/"
    print("User question:", user_question)
    # print out context
    print("Context provided to Gemini:")
    print(context)
    
    # Precise XML structure as requested
    xml_body = f"""
    <llm_request>
    <model>gemini-3-pro-preview</model>
    <system_prompt>
        You are a precise and technical assistant.
        Provide a coherent answer to the question only based on the provided context.
        Use the context tags to determine relevant information to use to answer the question.
    </system_prompt>
    <hyperparameters>
        <temperature>0</temperature>
        <top_p>0.1</top_p>
    </hyperparameters>
    <user_prompt>
    Context:
    {context}

    Question:
    {user_question}
        </user_prompt>
    </llm_request>
"""
    headers = {
        "Content-Type": "application/xml",
        "Authorization": f"Bearer {NALA_API_KEY}"
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.post(url, data=xml_body, headers=headers)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < retries:
                print(f"Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                return f"Error calling Gemini after {retries} attempts: {e}"


In [10]:
def run_rag_pipeline(user_query: str, top_k: int = 3, initial_fetch_k: int = 5):
    try:
        print(f"🔍 Processing Query: '{user_query}'")
        
        # --- Step 1: Initial Retrieval (Recall) ---
        query_vector = embedding_model.encode(user_query).tolist()
        session.execute(text("SET LOCAL hnsw.ef_search = 100;"))
        
        # Fetch more candidates (initial_fetch_k) than we need, to let the re-ranker filter them
        initial_results = session.query(
            DocumentChunk.content,
            Subtopic.subtopic_name,
            Topic.topic_name
        ).join(
            Subtopic, DocumentChunk.subtopic_id == Subtopic.id
        ).join(
            Topic, Subtopic.topic_id == Topic.id
        ).order_by(
            DocumentChunk.embedding.cosine_distance(query_vector)
        ).limit(initial_fetch_k).all()
        
        if not initial_results:
            return "No relevant documents found."

        # --- Step 2: Re-ranking (Precision) ---
        print(f"   > Re-ranking {len(initial_results)} candidates...")
        
        # Prepare pairs for the Cross-Encoder: [ (Query, Document_Text), ... ]
        pairs = [[user_query, row.content] for row in initial_results]
        
        # Predict scores (higher is better)
        scores = reranker.compute_score(pairs, normalize=True)

        scored_results = []
        for i, row in enumerate(initial_results):
            scored_results.append({
                'score': scores[i],
                'content': row.content,
                'subtopic_name': row.subtopic_name,
                'topic_name': row.topic_name
            })

        # Sort the structured list by score descending
        scored_results.sort(key=lambda x: x['score'], reverse=True)
        
        # Slice the top_k best chunks
        top_results = scored_results[:top_k]

        # --- Step 3: Build Context & Extract Metadata ---
        unique_topics = set()
        unique_subtopics = set()
        context_parts = []

        print(f"   > Selected Top {len(top_results)} after re-ranking:")
        
        for row in top_results:
            # Access data via dictionary keys instead of row.attributes
            unique_topics.add(row['topic_name'])
            unique_subtopics.add(row['subtopic_name'])
            print(f"     [{row['score']:.4f}] {row['topic_name']} > {row['subtopic_name']}")
            
            # Clean and format the content
            cleaned_content = row['content'].strip().replace("\n", " ")
            # remove excessive spaces
            cleaned_content = ' '.join(cleaned_content.split())
            context_parts.append(cleaned_content)

        # Join the cleaned content into a single context string
        full_context = "\n\n---\n\n".join(context_parts)
        
        # --- Step 4: Call Gemini & Parse JSON ---
        print("\n🤖 Asking Gemini...")
        raw_response_text = call_gemini_rag(full_context, user_query)
        
        # Parse the JSON string robustly
        try:
            response_json = json.loads(raw_response_text)
            clean_message = response_json.get("message", "Error: 'message' field not found.")
        except json.JSONDecodeError:
            clean_message = f"Raw output (not JSON): {raw_response_text}"

        # --- Step 5: Format Output ---
        topics_str = ", ".join(sorted(unique_topics))
        subtopics_str = ", ".join(sorted(unique_subtopics))
        
        final_output = (
            f"Relevant Subtopic(s): {subtopics_str} \n"
            f"Relevant Topic(s): {topics_str} \n"
            f"====================================\n"
            f"Possible Answer: {clean_message}"
        )
        
        return final_output

    except Exception as e:
        print(f"An error occurred in the pipeline: {e}")
        return f"Pipeline Error: {e}"
    finally:
        # Ensures DB connection is released
        session.close()

In [26]:
def run_subtopic_scan_pipeline(user_query: str, top_k: int = 5):
    try:
        print(f"🔍 Processing Query: '{user_query}'")
        
        # --- Step 1: Encode Query ---
        query_vector = embedding_model.encode(user_query).tolist()
        
        # --- Step 2: Retrieve Subtopics ---
        # Fetch subtopics based on cosine similarity with the query vector
        subtopic_results = session.query(
            Subtopic.subtopic_name,
            Topic.topic_name,
            Subtopic.subtopic_summary_embedding
        ).join(
            Topic, Subtopic.topic_id == Topic.id
        ).all()
        
        if not subtopic_results:
            return "No relevant subtopics found."

        # --- Step 3: Compute Similarity Scores ---
        print(f"   > Computing similarity scores for {len(subtopic_results)} subtopics...")
        
        scored_subtopics = []
        for row in subtopic_results:
            # Compute cosine similarity between query vector and subtopic embedding
            subtopic_embedding = row.subtopic_summary_embedding
            similarity_score = sum(q * s for q, s in zip(query_vector, subtopic_embedding))
            
            scored_subtopics.append({
                'score': similarity_score,
                'subtopic_name': row.subtopic_name,
                'topic_name': row.topic_name
            })

        # Sort the subtopics by similarity score in descending order
        scored_subtopics.sort(key=lambda x: x['score'], reverse=True)
        
        # Slice the top_k best subtopics
        top_results = scored_subtopics[:top_k]

        # --- Step 4: Format Output ---
        print(f"   > Selected Top {len(top_results)} subtopics:")
        
        for row in top_results:
            print(f"     [{row['score']:.4f}] {row['topic_name']} > {row['subtopic_name']}")

        # Prepare the final output
        topics_str = ", ".join(sorted(set(row['topic_name'] for row in top_results)))
        subtopics_str = ", ".join(sorted(set(row['subtopic_name'] for row in top_results)))
        
        final_output = (
            f"Relevant Subtopic(s): {subtopics_str} \n"
            f"Relevant Topic(s): {topics_str} \n"
            f"====================================\n"
        )
        
        return final_output

    except Exception as e:
        print(f"An error occurred in the pipeline: {e}")
        return f"Pipeline Error: {e}"
    finally:
        # Ensures DB connection is released
        session.close()

In [25]:
# Example Query (Change this to something relevant to your data)
query = "Give an industrial application on feedback control"

# Run the pipeline
response = run_rag_pipeline(query)

print("\n" + "="*30)
print("📝 FINAL RESPONSE")
print("="*30)
print(response)

🔍 Processing Query: 'Give an industrial application on feedback control'
   > Re-ranking 5 candidates...


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


   > Selected Top 3 after re-ranking:
     [0.3047] Introduction to Process Control > Feedback and Feedforward Control
     [0.1295] Dynamic Behaviour of First-Order and Second-Order Processes > Response of Second Order Processes
     [0.0517] Introduction to Process Control > Blending System Illustration

🤖 Asking Gemini...
User question: Give an industrial application on feedback control
Context provided to Gemini:
This chunk analyzes feedback (FB) control strategy by explaining its distinguishing feature, its advantages and disadvantages. The distinguishing feature of feedback control strategy is that it measures the controlled variable, and that it exhibits negative feedback in engineering applications, which is desirable for stability, in contrast to positive feedback common in social sciences. The advantages of feedback control are that corrective action is taken regardless of the source of the disturbance, and that it reduces sensitivity of the controlled variable to disturbance

In [27]:
query = "Give an industrial application of feedback control"

# Run the pipeline
response = run_subtopic_scan_pipeline(query)
print(response)

🔍 Processing Query: 'Give an industrial application of feedback control'
   > Computing similarity scores for 29 subtopics...
   > Selected Top 5 subtopics:
     [0.6175] Introduction to Process Control > Feedback and Feedforward Control
     [0.5222] Introduction to Process Control > Control Elements
     [0.5148] Introduction to Process Control > Blending System Illustration
     [0.5037] Introduction to Process Control > Elements of Control System
     [0.5009] Introduction to Process Control > Block Diagram
Relevant Subtopic(s): Blending System Illustration, Block Diagram, Control Elements, Elements of Control System, Feedback and Feedforward Control 
Relevant Topic(s): Introduction to Process Control 

